# 04 · Collaborative filtering: ItemKNN and ALS

Bar to beat (validation): **MostPopular(listeners, last 30d), recall@10 = 0.0511**.

Hypothesis from notebook 03: recency drives discovery, so CF trained on 4 years of history with equal
weight will struggle. Test it: every model is run with and without time decay
(each listen weighted by `0.5 ** (age_days / half_life_days)`).

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate
from musicrec.models.baselines import MostPopular
from musicrec.models.knn import ItemKNN
from musicrec.models.als import ALS

pd.set_option("display.float_format", "{:,.4f}".format)
split = Split.load(config.SPLIT_VAL_DIR)
COLS = ["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10", "fit_s"]

models = {}   # keep fitted models so we can re-evaluate without re-training

def run(model):
    t = time.time()
    model.fit(split)
    res = evaluate(model, split)
    res["fit_s"] = time.time() - t
    print(f"{model.name:<55} recall@10={res['recall@10']:.4f}  ({res['fit_s']:.0f}s)")
    models[model.name] = model
    return res

results = {}
pop = MostPopular(by="listeners", recent_days=30)
results[pop.name] = run(pop)

## ItemKNN

Plain cosine vs. cosine with shrinkage, each with and without time decay. ~1 min per model.

In [ ]:
for shrink in [0.0, 1.0]:   # relative shrink (see knn.py)
    for hl in [None, 30]:
        m = ItemKNN(k=100, shrink=shrink, half_life_days=hl)
        results[m.name] = run(m)

## ALS

Main sweep: half-life × alpha (confidence scale). A few seconds per model.
Note: `alpha` interacts with the weight scale. Decayed weights are small, so the best alpha shifts.

In [ ]:
for hl in [None, 14, 30, 90, 180]:
    for alpha in [2, 10]:
        m = ALS(factors=64, alpha=alpha, half_life_days=hl)
        results[m.name] = run(m)

In [ ]:
# Does model capacity help? Bigger embeddings at the best setting found above.
best_als = max((k for k in results if k.startswith("ALS")), key=lambda k: results[k]["recall@10"])
b = models[best_als]
print("best ALS so far:", best_als)
for f in [32, 128]:
    m = ALS(factors=f, alpha=b.alpha, half_life_days=b.half_life_days)
    results[m.name] = run(m)

In [ ]:
table = pd.DataFrame(results).T[COLS].sort_values("recall@10", ascending=False)
table.to_csv(config.PROJECT_ROOT / "reports" / "results_val_cf.csv")
table

## How much does recency matter? recall@10 vs. half-life (ALS)

In [ ]:
BLUE, GRAY = "#2a78d6", "#52514e"
rows = []
for hl in [None, 14, 30, 90, 180]:
    best = max(results[ALS(factors=64, alpha=a, half_life_days=hl).name]["recall@10"] for a in [2, 10])
    rows.append(("no decay" if hl is None else f"{hl}d", best))
labels, values = zip(*rows)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(labels, values, color=BLUE, lw=2, marker="o", markersize=8)
ax.axhline(results[pop.name]["recall@10"], color=GRAY, lw=1, ls="--")
ax.text(len(labels) - 1, results[pop.name]["recall@10"], "  Popular (last 30d)", va="bottom", ha="right", color=GRAY)
ax.set(xlabel="time-decay half-life", ylabel="recall@10 (validation)", title="ALS: recency weighting")
ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.3)
fig.savefig(config.FIGURES_DIR / "04_als_half_life.png", dpi=150, bbox_inches="tight")

## Robustness: discoveries that stuck (played ≥ 2 times)

In [ ]:
# Already-fitted models, re-evaluated with a stricter definition of "discovery"
best_knn = max((k for k in results if k.startswith("ItemKNN")), key=lambda k: results[k]["recall@10"])
best_als = max((k for k in results if k.startswith("ALS")), key=lambda k: results[k]["recall@10"])
stuck = {n: evaluate(models[n], split, min_plays=2) for n in [pop.name, best_knn, best_als]}
pd.DataFrame(stuck).T[["recall@10", "ndcg@10", "hit_rate@10", "n_eval_users"]]

## Segment analysis: new track by a known artist vs. a NEW artist

95% of listens are to artists the user already knows (EDA Q4). Are our "discoveries" mostly
deeper cuts from familiar artists? And which model is better at each kind?

In [ ]:
from musicrec.evaluation import artist_novelty
tracks = pd.read_parquet(config.TRACKS_PARQUET, columns=["track_key", "artist_name"])
new_artist = artist_novelty(split, tracks.set_index("track_key")["artist_name"])
print(f"share of discoveries from an artist the user already knows: {1 - new_artist.mean():.1%}")

seg = {}
for n in [pop.name, best_knn, best_als]:
    for label, mask in [("known artist", ~new_artist), ("new artist", new_artist)]:
        r = evaluate(models[n], split, targets=split.targets[mask])
        seg[(n, label)] = {"recall@10": r["recall@10"], "hit_rate@10": r["hit_rate@10"], "n_eval_users": r["n_eval_users"]}
pd.DataFrame(seg).T